# Theseus 教程（中文翻译版）

- 原始英文版：`05_differentiable_motion_planning.ipynb`
- 说明：本文件为自动翻译版本（保留代码不翻译，清空输出以减小体积）。如遇术语不一致，可优先参考英文原文。


# 运动规划第 2 部分：可微运动规划

在本教程中，我们将以运动规划教程的[第一部分](https://github.com/facebookresearch/theseus/blob/main/tutorials/04_motion_planning.ipynb)为基础，说明如何通过使用 `Theseus` 实现的运动规划器进行区分。特别是，我们将展示如何在 `torch` 中设置模仿学习循环，以生成初始化 `TheseusLayer` 的值，使其更快地收敛到高质量轨迹。如果您还没有这样做，我们鼓励您在继续本教程之前先查看运动规划教程的第 1 部分。

In [ ]:
import random 

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.data
from IPython.display import clear_output

import theseus as th
import theseus.utils.examples as theg

%load_ext autoreload
%autoreload 2

torch.set_default_dtype(torch.double)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.random.manual_seed(1)
random.seed(1)
np.random.seed(1)

mpl.rcParams["figure.facecolor"] = "white"
mpl.rcParams["font.size"] = 16

## 1. 初始设置

与运动规划教程的第 1 部分一样，第一步是从数据集中加载一些规划问题，并设置一些在整个实验中使用的常量。在此示例中，我们将使用从加载器获取的一批 2 个问题。

In [ ]:
dataset_dir = "data/motion_planning_2d"
num_prob = 2
dataset = theg.TrajectoryDataset(True, num_prob, dataset_dir, "tarpit")
data_loader = torch.utils.data.DataLoader(dataset, num_prob, shuffle=False)

batch = next(iter(data_loader))
map_size = batch["map_tensor"].shape[1]
trajectory_len = batch["expert_trajectory"].shape[2]
num_time_steps = trajectory_len - 1
map_size = batch["map_tensor"].shape[1]
safety_distance = 0.4
robot_radius = 0.4
total_time = 10.0
dt_val = total_time / num_time_steps
Qc_inv = [[1.0, 0.0], [0.0, 1.0]]
collision_w = 5.0
boundary_w = 100.0

接下来我们创建运动规划器。类 `theg.MotionPlanner` 存储按照第 1 部分中描述的步骤构造的 `TheseusLayer`，并且还提供一些有用的实用函数来从优化器的当前变量中检索轨迹。

In [ ]:
planner = theg.MotionPlanner(
    optimizer_config=("LevenbergMarquardt", {"max_optim_iters": 2, "step_size": 0.3}),
    map_size=map_size,
    epsilon_dist=safety_distance + robot_radius,
    total_time=total_time,
    collision_weight=collision_w,
    Qc_inv=Qc_inv,
    num_time_steps=num_time_steps,
    device=device,
)

由于我们正在处理单批数据，因此我们可以使用本示例中贯穿的一些张量来初始化运动规划器的输入字典。提醒一下，输入字典将 `TheseusLayer` 中的 `th.Variable` 名称与每个名称的张量值相关联。

In [ ]:
start = batch["expert_trajectory"][:, :2, 0]
goal = batch["expert_trajectory"][:, :2, -1]
planner_inputs = {
    "sdf_origin": batch["sdf_origin"].to(device),
    "start": start.to(device),
    "goal": goal.to(device),
    "cell_size": batch["cell_size"].to(device),
    "sdf_data": batch["sdf_data"].to(device),
}

## 2. 模仿学习循环

＃＃＃ 概述

在本例中，我们考虑以下模仿学习流程（参见第 2.2 节）：

* 对于一定数量的纪元：
    1. 从使用地图信息作为输入的 `nn.Module` 生成初始变量值（即轨迹）。
    2. 使用 `TheseusLayer` 规划轨迹，用步骤 1 的结果初始化优化变量。
    3. 计算一个损失，使步骤 2 的输出成为接近专家的高质量轨迹。
    4. 使用反向传播更新步骤1中模块的参数。

### 2.1。基本的初始轨迹模型

以下单元格创建用于生成初始轨迹的基本模型。该模型将地图 ID 的单热表示作为输入，并生成地图起始位置和目标位置之间的轨迹。输出是一个字典，其中键映射到变量名称，值映射到每个表示结果轨迹的初始值（张量）。

<div class="alertalert-blockalert-info">
<b>注意：</b>我们不包含模型代码，以便将重点放在学习部分，但感兴趣的读者可以在<a href="https://github.com/facebookresearch/theseus/blob/main/theseus/utils/examples/motion_planning/models.py">此处</a>找到它。该模型利用 GPMP2 的概率解释来轻松生成起始位置和目标位置之间的各种平滑轨迹。高级想法如下。轨迹的生成是首先产生一条抛物线，抛物线的焦点是起点和目标之间的中点，焦点和顶点之间的距离是MLP的输出；该抛物线使模型更容易学习宽曲线。然后，模型可以使用运动规划问题的概率公式，通过构建围绕抛物线的“采样”轨迹，向轨迹添加更高阶曲率。本例中的样本是另一个 MLP 的输出，它用作“采样”围绕抛物线的轨迹的种子。然后，模型返回该样本作为要使用的初始轨迹。</div>

In [ ]:
init_trajectory_model = theg.InitialTrajectoryModel(planner)
init_trajectory_model.to(device)
model_optimizer = torch.optim.Adam(init_trajectory_model.parameters(), lr=0.04) 

### 2.2。学习循环

模型到位后，我们现在可以将所有这些放在一起，通过运动规划器进行区分，并找到良好的初始轨迹以在两个地图上进行优化。该循环基本上遵循概述小节中的步骤 1-4。

In [ ]:
initial_trajectory_dicts = []
best_loss = float("inf")
losses = []
num_epochs = 100
best_epoch = None

# For speed considerations, we set the max number of optimizer iterations to a low number (2).
# This will also encourage the initial trajectory model to produce trajectories of higher quality.
planner.layer.optimizer.set_params(max_iterations=2)
for epoch in range(num_epochs):
    clear_output(wait=True)
    model_optimizer.zero_grad()
    
    # Step 1: Generate an initial trajectory by querying the model on the set of maps.
    initial_traj_dict = init_trajectory_model.forward(batch)
    # This updates the motion planner's input dictionary with the trajectories produced above.
    planner_inputs.update(initial_traj_dict)

    # Step 2: Optimize to improve on the initial trajectories produced by the model.
    planner.layer.forward(
        planner_inputs,
        optimizer_kwargs={
            "verbose": False,
            "damping": 0.1,
        }
    )        

    initial_trajectory_dicts.append(
        dict([(k, v.detach().clone()) for k, v in initial_traj_dict.items()]))

    # Step 3: Compute a loss evaluating the quality of the trajectories.
    # The loss consists of two terms. The first one encourages the final trajectory to 
    # match an expert trajectory available for this map (imitation_loss).
    # The second term uses the trajectory planner's total squared error, which also
    # encourages the trajectory to be smooth and avoid obstacles. We scale this term
    # by a small factor, since otherwise it would completely dominate over the
    # imitation loss.
    error_loss = planner.objective.error_metric().mean() / planner.objective.dim()

    solution_trajectory = planner.get_trajectory()
    imitation_loss = F.mse_loss(
        batch["expert_trajectory"].to(device), solution_trajectory)
    loss = imitation_loss + 0.001 * error_loss
    
    # Step 4: Do backpropagation through the TheseusLayer and update the model parameters.
    loss.backward()
    model_optimizer.step()
    
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_epoch = epoch
    losses.append(loss.item())
    print("------------------------------------")
    print(f"             Epoch {epoch}")
    print("------------------------------------")
    print(f"{'Imitation loss':20s}: {imitation_loss.item():.3f}")
    print(f"{'Error loss':20s}: {error_loss.item():.3f}")
    print(f"{'Total loss':20s}: {loss.item():.3f}")
    print("------------------------------------")
    print("------------------------------------")
    

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

## 3. 结果

现在让我们可视化使用学习到的初始化生成的轨迹，运行优化器进行几次迭代。以下函数可用于绘制变量值字典中的轨迹。

In [ ]:
def get_trajectory(values_dict):
    trajectory = torch.empty(values_dict[f"pose_0"].shape[0], 4, trajectory_len, device=device)
    for i in range(trajectory_len):
        trajectory[:, :2, i] = values_dict[f"pose_{i}"]
        trajectory[:, 2:, i] = values_dict[f"vel_{i}"]
    return trajectory

def plot_trajectories(initial_traj_dict, solution_traj_dict, include_expert=False):
    initial_traj = get_trajectory(initial_traj_dict).cpu()
    sol_traj = get_trajectory(solution_traj_dict).detach().clone().cpu()

    sdf = th.eb.SignedDistanceField2D(
        th.Point2(batch["sdf_origin"]),
        th.Variable(batch["cell_size"]),
        th.Variable(batch["sdf_data"]),
    )
    trajectories = [initial_traj, sol_traj]
    if include_expert:
        trajectories.append(batch["expert_trajectory"])
    figs = theg.generate_trajectory_figs(
        batch["map_tensor"], 
        sdf, 
        trajectories,
        robot_radius=0.4, 
        labels=["initial trajectory", "solution trajectory", "expert"], 
        fig_idx_robot=1,
        figsize=(6, 6)
    )
    for fig in figs:
        fig.show()

### 3.1。从直线初始化的轨迹

作为参考，下面我们展示了从直线初始化时经过 10 次优化器迭代后获得的轨迹质量。如图所示，直线产生的轨迹质量很差；需要超过 10 次迭代才能产生高质量的轨迹（在第 1 部分中，我们使用了 50 次）。

In [ ]:
straight_traj_dict = planner.get_variable_values_from_straight_line(
    planner_inputs["start"], planner_inputs["goal"])

planner_inputs.update(straight_traj_dict)
planner.layer.optimizer.set_params(max_iterations=10)
solution_dict, info = planner.layer.forward(
    planner_inputs,
    optimizer_kwargs={
        "verbose": False,
        "damping": 0.1,
    }
)
plot_trajectories(straight_traj_dict, solution_dict)

### 3.2 学习初始轨迹

另一方面，通过学习的初始轨迹，下图显示 10 次迭代足以产生避开所有障碍的平滑轨迹，说明了通过轨迹 planner.trj 进行区分的潜力

In [ ]:
planner_inputs.update(initial_trajectory_dicts[best_epoch])
planner.layer.optimizer.set_params(max_iterations=10)
solution_dict, info = planner.layer.forward(
    planner_inputs,
    optimizer_kwargs={
        "verbose": False,
        "damping": 0.1,
    }
)
plot_trajectories(
    initial_trajectory_dicts[best_epoch], solution_dict, include_expert=True)